# List of proposals

On this page you can download all of the data connected to a proposal where you have been added. The data will be zipped and downloaded. 

NB: we consider the following folder tree: `/mnt/<Instrument>_data/<proposal_id>`, where `<Instrument>` is one of the instrument (e.g. CAMEA, ICON), and `<proposal_id>` will be
automatically detected once you enter the previous path in `Path`.

In [ ]:
import os
import ipywidgets as ipw
from IPython.display import display
from IPython.display import FileLink
import shutil

# Widgets
search_box = ipw.Text(
    description='Path:',
    placeholder='Type or paste folder path here',
    value='/mnt'  # Default value
)

mount_label = ipw.HTML(value="No folder selected")

folder_dropdown = ipw.Dropdown(description='Instrument:', options=[])
subfolder_dropdown = ipw.Dropdown(description='ProposalID:', options=[])

subfolder_label = ipw.HTML(value="start")

download_button = ipw.Button(description="Download Selected", button_style='success', disabled=True)

# Store selected mount path
mount_path = ''

# Path to temporary zip file
temp_zip_path = './'

def on_search_submit(change):
    global mount_path
    mount_path = change['new']
    download_button.disabled = True
    subfolder_label.value = ""
    
    # Check if the path exists and is a directory
    if os.path.isdir(mount_path):
        mount_label.value = ""#Selected: {mount_path}"
        folders = [f.split('_')[0] for f in os.listdir(mount_path) if check_valid_instrument(mount_path, f)]
        
        if folders:
            folder_dropdown.options = folders 
            
        else:
            folder_dropdown.options = ['<No folders found>']
            subfolder_label.value = ""
    # If not valid, clear dropdowns and show error
    else:
        mount_label.value = "<p style='color: red'>Invalid folder path</p>"
        folder_dropdown.options = []
        subfolder_dropdown.options = []
        subfolder_label.value = ""

def check_valid_instrument(mount_path, folder):
    if not os.path.isdir(os.path.join(mount_path, folder)):
        return False
    
    # if we put a folder with no underscore, it will crash
    if len(folder.split('_')) < 2:
        return False
    
    if folder[0] == '.':
        return False
    if not folder.split('_')[1].lower() == 'data':
        return False
    return True



        
def on_folder_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        selected_folder = change['new']
        if selected_folder and selected_folder != '<No folders found>':
            path = os.path.join(mount_path, selected_folder+'_data')
            subfolders = [f for f in os.listdir(path) if check_valid_proposals(path, f)]
            if subfolders:
                subfolder_dropdown.options = subfolders
                download_button.disabled = False
            else:
                subfolder_dropdown.options = ['<No proposals found>']
                subfolder_label.value = ""
                download_button.disabled = True
        else:
            subfolder_dropdown.options = []
            subfolder_label.value = "No files found"

def check_valid_proposals(mount_path, folder):
    if not os.path.isdir(os.path.join(mount_path, folder)):
        return False
    if folder[0] == '.':
        return False
    if not len(folder) == 8:
        return False
    return True
    
    
def on_download_click(b):
    selected_main = folder_dropdown.value
    selected_sub = subfolder_dropdown.value
    if selected_main and selected_main != '<No folders found>':
        if selected_sub and selected_sub != '<No proposals found>':
            path_to_zip = os.path.join(mount_path, selected_main+'_data', selected_sub)
        else:
            path_to_zip = os.path.join(mount_path, selected_main)
        
        shutil.make_archive(subfolder_dropdown.value, 'zip', path_to_zip)
        display(FileLink(os.path.join(temp_zip_path,subfolder_dropdown.value+'.zip')))

def get_file_count():
    selected_main = folder_dropdown.value
    selected_sub = subfolder_dropdown.value
    if selected_main and selected_main != '<No folders found>':
        if selected_sub and selected_sub != '<No proposals found>':
            folder = os.path.join(mount_path, selected_main+'_data', selected_sub)
            return len([f for f in os.listdir(folder)])# if os.path.isfile(f) and f[0]!='.'])
    return -1

def on_proposal_changed(change):
    if change['type'] == 'change' and change['name'] == 'value':
        files = get_file_count()
        if files == 0:
           subfolder_label.value = 'No files found'
        else:
           subfolder_label.value = '{:} file{:} found'.format(files,'s'*(files>1))
        if files < 0:
            subfolder_label.value = ""


        
# Bind events
search_box.observe(on_search_submit, names='value')
folder_dropdown.observe(on_folder_change)
subfolder_dropdown.observe(on_proposal_changed)
download_button.on_click(on_download_click)


# Display widgets
display(ipw.VBox([
    ipw.HBox([search_box, mount_label]),
    folder_dropdown,
    ipw.HBox([subfolder_dropdown,subfolder_label]),
    download_button
]))


## To initialize
mockup = {'new':search_box.value}
on_search_submit(mockup)